# Week 04 — Baseline Action Score & Top-10 Review

**Course:** FlyRank AI Fluency Track  
**Phase:** Machine Learning Foundations  
**Module:** Baseline Ranking Model, Action Scoring, and Top-10 Audit  

---

## Section 1: Baseline Problem Framing & Metric Setup

* **Goal:** Rank content URLs by CTR / Engagement Opportunity score to prioritize pages requiring metadata rewrites.
* **Baseline Model:** Heuristic Scoring & Simple Logistic Regression.
* **Primary Metric:** Precision@10 (Percentage of top 10 ranked URLs that represent true high-value opportunities).

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Load / simulate features dataframe from mid-panel month
np.random.seed(42)
n_samples = 200

urls = [f"https://flyrank.ai/blog/guide-{i}" for i in range(1, n_samples + 1)]
impressions = np.random.randint(500, 25000, size=n_samples)
clicks = (impressions * np.random.uniform(0.005, 0.06, size=n_samples)).astype(int)
ctr = clicks / impressions
avg_position = np.random.uniform(1.2, 18.0, size=n_samples)

# True Label: High Impressions (> 3000) but Low CTR (< 2%)
y_true = ((impressions > 3000) & (ctr < 0.02)).astype(int)

df = pd.DataFrame({
    'url': urls,
    'impressions': impressions,
    'clicks': clicks,
    'ctr': ctr,
    'avg_position': avg_position,
    'is_opportunity': y_true
})

print(f"Total Pages Analyzed: {len(df)} | Total Opportunities Found: {y_true.sum()}")
df.head()

Total Pages Analyzed: 200 | Total Opportunities Found: 46


,url,impressions,clicks,ctr,avg_position,is_opportunity
0,https://flyrank.ai/blog/guide-1,24154,1373,0.056844,1.972844,0
1,https://flyrank.ai/blog/guide-2,16295,371,0.022768,1.884244,0
2,https://flyrank.ai/blog/guide-3,1360,45,0.033088,15.571738,0
3,https://flyrank.ai/blog/guide-4,5890,257,0.043633,13.021452,0
4,https://flyrank.ai/blog/guide-5,22075,551,0.024960,9.166120,0


## Section 2: Baseline Scoring & Model Execution

We calculate a **Baseline Heuristic Score** (`impressions / avg_position`) and compare it against a **Logistic Regression Action Scorer**.

In [2]:
# 1. Heuristic Baseline Score
df['heuristic_score'] = df['impressions'] / (df['avg_position'] + 1.0)

# 2. Logistic Regression Baseline Scorer
X = df[['impressions', 'ctr', 'avg_position']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(random_state=42)
model.fit(X_scaled, y_true)
df['model_action_score'] = model.predict_proba(X_scaled)[:, 1]

print("Scoring Complete. Comparing Top 10 Predictions...")

Scoring Complete. Comparing Top 10 Predictions...


## Section 3: Top-10 Opportunities Audit & Precision@10 Evaluation

In [3]:
# Top 10 Ranked by Model Action Score
top_10_model = df.sort_values(by='model_action_score', ascending=False).head(10)

# Calculate Precision@10
precision_at_10 = top_10_model['is_opportunity'].sum() / 10.0
print(f"🎯 [EVALUATION] Baseline Model Precision@10: {precision_at_10 * 100:.1f}%")

print("\n--- TOP 10 ACTION OPPORTUNITIES RANKED BY MODEL ---")
top_10_model[['url', 'impressions', 'ctr', 'avg_position', 'model_action_score', 'is_opportunity']]

🎯 [EVALUATION] Baseline Model Precision@10: 100.0%

--- TOP 10 ACTION OPPORTUNITIES RANKED BY MODEL ---


,url,impressions,ctr,avg_position,model_action_score,is_opportunity
37,https://flyrank.ai/blog/guide-38,24618,0.005890,1.506127,0.978657,1
110,https://flyrank.ai/blog/guide-111,22502,0.005822,14.855033,0.977794,1
157,https://flyrank.ai/blog/guide-158,22903,0.007030,11.618228,0.970663,1
11,https://flyrank.ai/blog/guide-12,22462,0.006990,11.546282,0.969644,1
67,https://flyrank.ai/blog/guide-68,24439,0.009616,16.362889,0.956455,1
151,https://flyrank.ai/blog/guide-152,17747,0.006931,16.339455,0.955363,1
198,https://flyrank.ai/blog/guide-199,14045,0.005767,17.943499,0.952167,1
149,https://flyrank.ai/blog/guide-150,24742,0.010145,11.742976,0.949791,1
178,https://flyrank.ai/blog/guide-179,14431,0.006375,16.869839,0.946442,1
56,https://flyrank.ai/blog/guide-57,23119,0.010078,17.840486,0.946066,1


## Section 4: Qualitative Review of Top-10 Recommendations

1. **Recommendation Accuracy:** All top 10 recommended URLs exhibit high impression counts (> 5,000) coupled with sub-2% CTRs, making them ideal candidates for title tag and meta description rewrites.
2. **Action Code:** `REWRITE_META_DESCRIPTION` & `TEST_HEADLINE_HOOK`.
3. **Baseline Limitation:** The logistic model treats all positions linearly. Non-linear models (XGBoost / Random Forests) will be implemented in Week 5 to capture position rank drops.

## Section 5: Self-Check Checklist

- [x] **Baseline Model Executed:** Logistic Regression & Heuristic scoring run.  
- [x] **Top-10 Recommendations Displayed:** Dataframe table output verified.  
- [x] **Precision@10 Metric Calculated:** Evaluated against ground-truth labels.  
- [x] **Qualitative Audit Stated:** Specific reason codes assigned to top recommendations.  